<a href="https://colab.research.google.com/github/dpalma1996/Derian_Palma_Q-Data-Analytics-2026-2027/blob/main/Tasca%20S7.01.%20Creaci%C3%B3%20de%20funcions%2C%20estructures%20de%20dades%20i%20bucles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tasca S7.01. Creació de funcions, estructures de dades i bucles
**Autor: Derian Palma Quirós**

Este notebook resuelve seis ejercicios mediante funciones, diccionarios, listas y bucles. En cada ejercicio describo el algoritmo, justifico las decisiones principales y presento una ejecución reproducible.

Las explicaciones, mensajes y docstrings están en castellano. Se conservan los nombres de funciones y parámetros exigidos y las claves técnicas en catalán del material de la academia para mantener su correspondencia con los enunciados.

## Cómo ejecutar
Ejecutar las celdas de arriba abajo. La prueba de selección simula `input()` para evitar solicitudes manuales durante una ejecución completa. Para usar el formulario manualmente, ejecutar `candidatos = main_seleccio()` en otra celda.

Para el último ejercicio, colocar `historic partits.txt` junto al notebook; en Colab, subirlo a `/content`.

In [45]:
import string
from pathlib import Path
from unittest.mock import patch

import numpy as np

## Nivell 1

### Exercici 1: Selecció de Personal

**Objetivo:** registrar diez candidatos válidos, mostrar su puntuación individual y presentar el informe por puntuación descendente.

#### Algoritmo
1. `main_seleccio` crea una lista vacía y repite el proceso hasta tener diez candidatos.
2. `obtenir_dades` solicita los seis campos. La auxiliar `pedir_dato_valido` consulta `validar_dades` inmediatamente y repite solo el campo incorrecto.
3. El nombre debe contener alguna letra y solo admite letras, espacios, guion y apóstrofo recto. El teléfono se conserva como texto de nueve dígitos ASCII. La experiencia se convierte a entero no negativo; las tres escalas se convierten a enteros entre 0 y 3.
4. `calcular_puntuacio` devuelve experiencia × 1,5 + catalán + Excel + titulación.
5. `mostrar_individual` muestra nombre y puntuación. `classificar_candidat` devuelve prioridad alta (≥ 12), adecuado (≥ 8 y < 12) o no prioritario (< 8).
6. La coordinadora añade puntuación y categoría al diccionario y lo incorpora a la lista. La longitud de la lista cuenta las inscripciones válidas.
7. `mostrar_informe` muestra nombre, teléfono y puntuación ordenados de mayor a menor. La coordinadora devuelve la lista.

**Decisiones:** nombre completo en un único campo; teléfono como identificador; experiencia en años completos sin máximo definido. Ordenar por puntuación también ordena las categorías, porque dependen de sus umbrales. Los empates conservan el orden de inscripción. Las claves abreviadas del diccionario original se mantienen: `nom`, `tel`, `exp`, `cat`, `exc`, `tit`, `punt` y `clas`.

**Evolución del planteamiento:** inicialmente propuse teléfono entero, un diccionario para todo el conjunto e informe en orden de entrada. Lo corregí a teléfono de texto, lista de diccionarios y orden descendente; además elegí validar cada campo antes de avanzar.

In [80]:
inputs_test = [
    "Laura Serra", "608112233", 6, 3, 3, 3,
    "Clara Gómez", "655221199", 3, 3, 3, 2,
    "Sergio López", "644112211", 3, 2, 2, 2,
    "Candidat Error", "611000000", 5, 7, 3, 2, # Català = 7 → NO VÀLID
    "Marta Ruiz", "699887766", 2, 3, 2, 1,
    "Marc Vidal", "612443322", 6, 3, 3, 3,
    "Júlia Romero", "677998877", 5, 3, 3, 2,
    "Pol Navarro", "699554433", 4, 3, 2, 2,
    "David Ferrer", "677665544", 2, 1, 1, 1,
    "Anna Paredes", "633221144", 1, 2, 1, 0,
    "Anibal Paredes", "644272727", 1, 2, 1, 0,
    "Núria Casals", "600112200", 8, 2, 3, 3 # 11è candidat
]

In [81]:
def validar_dades(dato, campo):
    """Comprueba si un dato cumple las reglas del campo indicado."""

    if campo == "telefon":
        return dato.isascii() and dato.isdigit() and len(dato) == 9

    elif campo == "experiencia":
        try:
            dato = int(dato)
            return dato >= 0
        except ValueError:
            return False

    elif campo in ("catalan", "excel", "titulo"):
        try:
            dato = int(dato)
            return 0 <= dato <= 3
        except ValueError:
            return False

    elif campo == "nombre":
        if dato.strip() == "":
            return False

        letra = False

        for caracter in dato:
            if caracter.isalpha():
                letra = True

            if not (
                caracter.isalpha()
                or caracter == " "
                or caracter == "'"
                or caracter == "-"
            ):
                return False

        return letra

    else:
        return False

In [82]:
def obtenir_dades():
    """Recoge los seis campos de un registro sin convertirlos."""

    candidat = {}

    print("""

    **********

    """)
    candidat["nom"] = input("Introduzca el nombre y apellidos: ")

    candidat["tel"] = input(
        "Introduzca el número de teléfono de contacto: "
    )

    candidat["exp"] = input(
        "Introduzca los años de experiencia dentro de la empresa: "
    )

    candidat["cat"] = input("""Introduzca el conocimiento de catalán:
0 = sin conocimiento
1 = hasta el nivel B1
2 = nivel B2
3 = C1 o superior
Seleccione una opción: """)

    candidat["exc"] = input("""Introduzca el dominio de Excel:
0 = nivel básico
1 = uso de funciones
2 = tablas dinámicas
3 = Visual Basic y macros
Seleccione una opción: """)

    candidat["tit"] = input("""Introduzca el título académico:
0 = ESO
1 = FP
2 = Universidad
3 = Máster o superior
Seleccione una opción: """)

    return candidat

In [83]:
def calcular_puntuacio(cand):
    """Calcula la puntuación del candidato."""

    punt = cand["exp"]*1.5 + cand["cat"] + cand["exc"] + cand["tit"]

    return punt

In [84]:
def classificar_candidat(punt):
    """Asigna una categoría según la puntuación."""

    if punt >= 12:
        clas = "Prioridad Alta"
    elif punt >= 8:
        clas = "Candidato adecuado"
    else:
        clas = "No prioritario"

    return clas

In [85]:
def mostrar_individual(cand):
    """Muestra la puntuación individual de un candidato válido."""

    print(f'{cand["nom"]} - {cand["punt"]} puntos')

In [86]:
def mostrar_informe(lcand):
    """Muestra los candidatos ordenados por puntuación descendente."""

    informe = sorted(lcand, key=lambda cand: cand["punt"], reverse=True)

    print("\nINFORME FINAL")

    for cand in informe:
        print(
            f'{cand["nom"]} - {cand["tel"]} - '
            f'{cand["punt"]} puntos'
        )

In [87]:
def main_seleccio():
    """Registra diez candidatos válidos y conserva las incidencias."""

    lcand = []
    incidencias = []

    campos = {
        "nom": "nombre",
        "tel": "telefon",
        "exp": "experiencia",
        "cat": "catalan",
        "exc": "excel",
        "tit": "titulo"
    }

    while len(lcand) < 10:
        cand = obtenir_dades()
        errores = []

        for clave, campo in campos.items():
            if not validar_dades(cand[clave], campo):
                errores.append(campo)

        if errores:
            incidencias.append({
                "registro": cand.copy(),
                "campos_invalidos": errores
            })

            print(
                f'Registro no admitido: {cand["nom"]}. '
                f'Campos incorrectos: {", ".join(errores)}'
            )

        else:
            for campo in ("exp", "cat", "exc", "tit"):
                cand[campo] = int(cand[campo])

            cand["punt"] = calcular_puntuacio(cand)
            cand["clas"] = classificar_candidat(cand["punt"])

            lcand.append(cand)
            mostrar_individual(cand)

    mostrar_informe(lcand)

    return lcand

#### Prueba con entrada de datos simulada

Para agilizar la prueba, se utiliza la lista `inputs_test` como fuente
de respuestas, evitando introducir cada dato manualmente.

La función auxiliar `input_simulado` se utiliza únicamente en la prueba:
devuelve el siguiente valor de la lista como texto y muestra la pregunta
junto con la respuesta. Mediante `patch`, sustituye temporalmente la
entrada por teclado.

El programa ejecuta las funciones reales de selección: recoge los datos,
los valida, calcula la puntuación, clasifica a los candidatos y presenta
el informe. La función `main_seleccio` controla cuándo se alcanzan los
diez registros válidos. Al finalizar el bloque, `input()` recupera su
funcionamiento habitual.

La lista contiene doce registros, uno de ellos con un valor de catalán
no permitido (7). Esta ejecución permite observar cómo se descarta ese
registro y se continúa con el siguiente.

In [88]:
respuestas_test = iter(inputs_test)

def input_simulado(mensaje=""):
    """Toma la siguiente respuesta de prueba y la muestra."""

    try:
        respuesta = str(next(respuestas_test))
    except StopIteration:
        raise RuntimeError(
            "Se agotaron los datos de prueba antes de completar "
            "los diez candidatos válidos."
        ) from None

    print(mensaje + respuesta + "\n")
    return respuesta


with patch("builtins.input", side_effect=input_simulado):
    candidatos_test = main_seleccio()

print("\nProceso finalizado.")


    
    **********
    
    
Introduzca el nombre y apellidos: Laura Serra

Introduzca el número de teléfono de contacto: 608112233

Introduzca los años de experiencia dentro de la empresa: 6

Introduzca el conocimiento de catalán:
0 = sin conocimiento
1 = hasta el nivel B1
2 = nivel B2
3 = C1 o superior
Seleccione una opción: 3

Introduzca el dominio de Excel:
0 = nivel básico
1 = uso de funciones
2 = tablas dinámicas
3 = Visual Basic y macros
Seleccione una opción: 3

Introduzca el título académico:
0 = ESO
1 = FP
2 = Universidad
3 = Máster o superior
Seleccione una opción: 3

Laura Serra - 18.0 puntos

    
    **********
    
    
Introduzca el nombre y apellidos: Clara Gómez

Introduzca el número de teléfono de contacto: 655221199

Introduzca los años de experiencia dentro de la empresa: 3

Introduzca el conocimiento de catalán:
0 = sin conocimiento
1 = hasta el nivel B1
2 = nivel B2
3 = C1 o superior
Seleccione una opción: 3

Introduzca el dominio de Excel:
0 = nivel básico
1 =

**Resultado:** se registran diez candidatos válidos. Laura y Marc
encabezan el informe con 18 puntos, seguidos de Júlia con 15,5 puntos.

El registro «Candidat Error» no se admite porque el valor de catalán
es 7, fuera del rango permitido de 0 a 3. El programa informa del error
y continúa con el siguiente registro.

Anibal completa los diez candidatos válidos. Los datos de Núria
no se solicitan porque ya se ha alcanzado el objetivo.

### Exercici 2: Segmentació per risc de cistella de compra

#### Algoritmo
1. `main_marketing` entrega la lista a `processar_clients`.
2. Para cada cliente, se copia el diccionario original.
3. `obtenir_segment_i_factor` consulta un diccionario de diccionarios por código y devuelve segmento, mensaje y factor.
4. `obtenir_factor_edat` consulta el factor del grupo de edad ya proporcionado.
5. `calcular_risc` multiplica los factores. Se añaden todos los campos al diccionario y se guarda por identificador.
6. `mostrar_diccionari` presenta el resultado y `mostrar_respostes` responde las cuatro preguntas de marketing. La principal devuelve los resultados.

**Decisiones:** la segmentación se resuelve sin condiciones. Las etiquetas y mensajes son propuestas propias; los factores son los del enunciado. El riesgo es un índice, no una probabilidad: puede superar 1. Las claves originales `id_client`, `nom`, `edat`, `grup_edat` y `codi_comportament` corresponden a los campos identificador, nombre, edad, grupo y código pedidos en el resultado.

Una prioridad alta según este índice no demuestra rentabilidad: faltan margen, coste de contacto y probabilidad de conversión.

In [89]:
clients_marketing = [
    {"id_client": 1, "nom": "Laura Serra", "edat": 23, "grup_edat": "18-25", "codi_comportament": 4},
    {"id_client": 2, "nom": "Marc Vidal", "edat": 35, "grup_edat": "26-40", "codi_comportament": 3},
    {"id_client": 3, "nom": "Ana López", "edat": 29, "grup_edat": "26-40", "codi_comportament": 2},
    {"id_client": 4, "nom": "Joan Riera", "edat": 19, "grup_edat": "18-25", "codi_comportament": 2},
    {"id_client": 5, "nom": "Mónica Pérez", "edat": 47, "grup_edat": "41-60", "codi_comportament": 4},
    {"id_client": 6, "nom": "Luis García", "edat": 52, "grup_edat": "41-60", "codi_comportament": 1},
    {"id_client": 7, "nom": "Pilar Sánchez", "edat": 61, "grup_edat": "60+", "codi_comportament": 3},
    {"id_client": 8, "nom": "Eva Martín", "edat": 38, "grup_edat": "26-40", "codi_comportament": 0},
    {"id_client": 9, "nom": "Diego Romero", "edat": 24, "grup_edat": "18-25", "codi_comportament": 1},
    {"id_client": 10, "nom": "Núria Costa", "edat": 33, "grup_edat": "26-40", "codi_comportament": 4}
]

In [90]:
def obtenir_factor_edat(grup_edat):
    """Consulta el factor asignado al grupo de edad."""

    factors = {
      "18-25": 1.20,
      "26-40": 1.00,
      "41-60": 0.85,
      "60+": 0.70
    }

    return factors[grup_edat]

In [91]:
def obtenir_segment_i_factor(codi):
    """Consulta segmento, mensaje y factor mediante el código de comportamiento."""

    comportaments = {
      0: {
        "segment": "Navega sin añadir productos",
        "missatge": "Descubre productos que podrían interesarte.",
        "factor_codi": 0.10
      },
      1: {
        "segment": "Añade productos sin llegar al checkout",
        "missatge": "¿Has encontrado lo que buscabas? Revisa tu cesta y continúa tu compra.",
        "factor_codi": 0.25
      },
      2: {
        "segment": "Llega al checkout pero no paga",
        "missatge": "¿Necesitas ayuda para finalizar tu compra? Retoma el proceso de pago.",
        "factor_codi": 0.50
      },
      3: {
        "segment": "Cesta abandonada hace menos de 24 horas",
        "missatge": "¿Olvidaste algo? Vuelve a tu cesta y completa tu compra.",
        "factor_codi": 0.75
      },
      4: {
        "segment": "Cesta abandonada hace más de 24 horas",
        "missatge": "¿Sigues interesado? Revisa los productos de tu cesta y retoma tu compra.",
        "factor_codi": 0.90
      }
    }

    return comportaments[codi]

In [92]:
def calcular_risc(factor_codi, factor_edat):
    """Multiplica los factores para obtener el índice de riesgo del ejercicio."""

    return factor_codi * factor_edat

In [93]:
def processar_clients(clients):
    """Enriquece copias de los clientes y las devuelve indexadas por identificador."""

    resultat = {}
    for client in clients:
        dades = client.copy()
        comportament = obtenir_segment_i_factor(client['codi_comportament'])
        factor_edat = obtenir_factor_edat(client['grup_edat'])
        dades['segment'] = comportament['segment']
        dades['missatge'] = comportament['missatge']
        dades['factor_codi'] = comportament['factor_codi']
        dades['factor_edat'] = factor_edat
        dades['risc_final'] = calcular_risc(dades['factor_codi'], dades['factor_edat'])
        resultat[client['id_client']] = dades

    return resultat

In [94]:
def mostrar_diccionari(resultat):
    """Muestra todos los campos de los clientes procesados."""

    formatos = {float: "{:.4f}".format}

    print("\nRESULTADOS POR CLIENTE")

    for dades in resultat.values():
        print("\n------------------------------")

        for camp, valor in dades.items():
            print(f"{camp}: {formatos.get(type(valor), str)(valor)}")

In [95]:
def mostrar_respostes(resultat):
    """Presenta las cuatro respuestas de marketing basadas en el índice y segmento."""

    ordenats = sorted(resultat.values(), key=lambda client: client['risc_final'], reverse=True)
    print('\n1. ¿A quién contactar primero?')
    print('Priorizamos de mayor a menor índice de riesgo:')
    for client in ordenats:
        print(f"{client['nom']}: {client['risc_final']:.4f}")
    print('\n2. Mismo comportamiento y distintas edades')
    grups = {}
    for client in resultat.values():
        codi = client['codi_comportament']
        grups.setdefault(codi, []).append(client)
    for codi, membres in sorted(grups.items()):
        print(f'\nCódigo {codi}:')
        for client in membres:
            print(f"{client['nom']} | Grupo: {client['grup_edat']} | Factor edad: {client['factor_edat']} | Riesgo: {client['risc_final']:.4f}")
    print('\nCon el mismo comportamiento, un factor de edad mayor produce un riesgo mayor según las reglas del ejercicio.')
    print('\n3. ¿Quién presenta menor interés?')
    print('El código 0 corresponde a quienes navegan sin añadir productos.')
    for client in grups.get(0, []):
        print(f"{client['nom']} | {client['segment']} | Riesgo: {client['risc_final']:.4f}")
    print('Este segmento tendría menor prioridad para invertir en recuperación de cestas, porque todavía no ha creado una.')
    print('\n4. ¿Qué cliente sería más rentable recuperar?')
    prioritari = ordenats[0]
    print(f"{prioritari['nom']} tiene el mayor índice de riesgo ({prioritari['risc_final']:.4f}) y pertenece al segmento: {prioritari['segment']}.")
    print('Lo proponemos como primera prioridad según este índice. Sin embargo, no podemos asegurar que sea el más rentable: faltan el margen de la compra, el coste de contacto y la probabilidad de recuperación.')

In [97]:
def main_marketing(clients):
    """Coordina el procesamiento y la presentación del análisis de marketing."""

    resultat = processar_clients(clients)
    mostrar_diccionari(resultat)
    mostrar_respostes(resultat)

    return resultat

In [36]:
resultat_marketing = main_marketing(clients_marketing)


RESULTADOS POR CLIENTE

------------------------------
id_client: 1
nom: Laura Serra
edat: 23
grup_edat: 18-25
codi_comportament: 4
segment: Cesta abandonada hace más de 24 horas
missatge: ¿Sigues interesado? Revisa los productos de tu cesta y retoma tu compra.
factor_codi: 0.9000
factor_edat: 1.2000
risc_final: 1.0800

------------------------------
id_client: 2
nom: Marc Vidal
edat: 35
grup_edat: 26-40
codi_comportament: 3
segment: Cesta abandonada hace menos de 24 horas
missatge: ¿Olvidaste algo? Vuelve a tu cesta y completa tu compra.
factor_codi: 0.7500
factor_edat: 1.0000
risc_final: 0.7500

------------------------------
id_client: 3
nom: Ana López
edat: 29
grup_edat: 26-40
codi_comportament: 2
segment: Llega al checkout pero no paga
missatge: ¿Necesitas ayuda para finalizar tu compra? Retoma el proceso de pago.
factor_codi: 0.5000
factor_edat: 1.0000
risc_final: 0.5000

------------------------------
id_client: 4
nom: Joan Riera
edat: 19
grup_edat: 18-25
codi_comportament: 2
s

**Interpretación:** Laura (1,0800), Núria (0,9000) y Mónica (0,7650) son las primeras por índice. Comparten código 4, pero sus factores de edad difieren. Eva, código 0 e índice 0,1000, es la de menor interés observado y menor prioridad para recuperar una cesta. Laura es la primera por la regla adoptada, pero no puede afirmarse que sea la más rentable.

## Nivell 2

### Exercici 1: Gestió de risc i priorització de pacients

#### Algoritmo
1. `convertir_pacient` separa cada registro por `|`, limpia el nombre y convierte la edad a entero.
2. `normalitzar_simptomes` divide por comas, pasa a minúsculas y une las palabras con guiones bajos.
3. `calcular_risc_total` suma los valores del diccionario para los síntomas normalizados.
4. `processar_pacients` repite la conversión y el cálculo para todos los registros.
5. `ordenar_pacients` ordena por riesgo descendente; `mostrar_pacients` presenta las filas.
6. `main_pacients` coordina las funciones y devuelve un diccionario indexado por posición.

**Decisiones:** la edad no entra en la fórmula. Los empates mantienen el orden original. Un síntoma desconocido provoca `KeyError` en lugar de asignarle silenciosamente riesgo cero; los datos facilitados contienen solo síntomas conocidos.

In [1]:
pacients = [
    "Maria|45|tos, Fiebre Alta, dolor_pecho",
    "Luis|33|dolor_cabeza,faTiga",
    "Sara|67|fiebre alta, dificultad RESPIRAR, tos",
    "Jordi|52|fatiga , tos",
    "Anna|29|fiebre alta,dolor_cabeza"
]

In [2]:
nivells_risc = {
    "fiebre_alta": 3,
    "dificultad_respirar": 5,
    "dolor_pecho": 4,
    "tos": 1,
    "fatiga": 1,
    "dolor_cabeza": 1
}

In [14]:
def normalitzar_simptomes(text):
    """Separa síntomas por comas y normaliza mayúsculas y espacios."""

    simptomes = []
    for simptom in text.split(','):
        net = '_'.join(simptom.lower().split())
        simptomes.append(net)

    return simptomes

In [15]:
def convertir_pacient(registre):
    """Convierte un registro en un diccionario con edad entera y síntomas limpios."""

    nom, edat, text_simptomes = registre.split('|')
    pacient = {'nom': nom.strip(), 'edat': int(edat), 'simptomes': normalitzar_simptomes(text_simptomes)}

    return pacient

In [16]:
def calcular_risc_total(simptomes, nivells_risc):
    """Suma los valores de los síntomas según el diccionario del ejercicio."""

    total = 0
    for simptom in simptomes:
        total += nivells_risc[simptom]

    return total

In [17]:
def processar_pacients(pacients, nivells_risc):
    """Convierte los registros y añade su riesgo total."""

    resultat = []

    for registre in pacients:
        pacient = convertir_pacient(registre)
        pacient['risc_total'] = calcular_risc_total(pacient['simptomes'], nivells_risc)
        resultat.append(pacient)

    return resultat

In [19]:
def ordenar_pacients(pacients):
    """Ordena pacientes por riesgo descendente conservando el orden en los empates."""

    return sorted(pacients, key=lambda pacient: pacient['risc_total'], reverse=True)

In [18]:
def mostrar_pacients(pacients):
    """Muestra nombre, edad, síntomas y riesgo de cada paciente."""

    for pacient in pacients:
        print(f"{pacient['nom']} | Edad: {pacient['edat']} | Síntomas: {', '.join(pacient['simptomes'])} | Riesgo total: {pacient['risc_total']}")

In [22]:
def main_pacients(pacients, nivells_risc):
    """Coordina el procesamiento y devuelve pacientes indexados por prioridad."""

    processats = processar_pacients(pacients, nivells_risc)

    print('PACIENTES PROCESADOS')
    mostrar_pacients(processats)

    ordenats = ordenar_pacients(processats)
    print('\nPRIORIDAD SEGÚN EL RIESGO DEL EJERCICIO')
    mostrar_pacients(ordenats)

    resultat = {}
    for posicio, pacient in enumerate(ordenats, start=1):
        resultat[posicio] = pacient

    return resultat

In [23]:
resultat_pacients = main_pacients(pacients, nivells_risc)

PACIENTES PROCESADOS
Maria | Edad: 45 | Síntomas: tos, fiebre_alta, dolor_pecho | Riesgo total: 8
Luis | Edad: 33 | Síntomas: dolor_cabeza, fatiga | Riesgo total: 2
Sara | Edad: 67 | Síntomas: fiebre_alta, dificultad_respirar, tos | Riesgo total: 9
Jordi | Edad: 52 | Síntomas: fatiga, tos | Riesgo total: 2
Anna | Edad: 29 | Síntomas: fiebre_alta, dolor_cabeza | Riesgo total: 4

PRIORIDAD SEGÚN EL RIESGO DEL EJERCICIO
Sara | Edad: 67 | Síntomas: fiebre_alta, dificultad_respirar, tos | Riesgo total: 9
Maria | Edad: 45 | Síntomas: tos, fiebre_alta, dolor_pecho | Riesgo total: 8
Anna | Edad: 29 | Síntomas: fiebre_alta, dolor_cabeza | Riesgo total: 4
Luis | Edad: 33 | Síntomas: dolor_cabeza, fatiga | Riesgo total: 2
Jordi | Edad: 52 | Síntomas: fatiga, tos | Riesgo total: 2


**Resultado:** Sara obtiene 9 puntos, Maria 8 y Anna 4. Luis y Jordi empatan a 2; mantienen su orden de entrada. Los cinco registros quedan normalizados.

### Exercici 2: Classificació de clients de segurs

#### Algoritmo
1. `obtenir_nivell_seguro` consulta el código sin `if/elif`.
2. `calcular_preu_final` aplica precio base × (1 + edad / 100).
3. `processar_clients_seguro` añade categoría, precio base, fraude y precio final a una copia de cada cliente.
4. `ordenar_clients_seguro` ordena por fraude descendente y después por precio descendente.
5. `mostrar_clients_seguro` presenta los resultados; `analitzar_clients_seguro` identifica los máximos y la prioridad de revisión.
6. `main_seguro` coordina y devuelve un diccionario por posición.

**Decisiones:** se interpreta la ordenación como dos criterios sucesivos; el precio descendente desempata el fraude.

In [24]:
clients_seguro = [
    {"nom": "Ana", "edat": 42, "codi": 0},
    {"nom": "Carlos", "edat": 30, "codi": 2},
    {"nom": "Isabel", "edat": 55, "codi": 3},
    {"nom": "Jorge", "edat": 40, "codi": 1},
    {"nom": "Marta", "edat": 28, "codi": 0}
]

In [25]:
nivells_seguro = {
    0: {"categoria": "Baix risc", "preu_base": 120, "fraude": 0.01},
    1: {"categoria": "Risc mitjà", "preu_base": 200, "fraude": 0.05},
    2: {"categoria": "Risc alt", "preu_base": 350, "fraude": 0.15},
    3: {"categoria": "Risc crític", "preu_base": 500, "fraude": 0.30}
}

In [39]:
def obtenir_nivell_seguro(codi, nivells_seguro):
    """Consulta la categoría, el precio base y el fraude asociados al código."""

    return nivells_seguro[codi]

In [38]:
def calcular_preu_final(preu_base, edat):
    """Calcula el precio de la póliza aplicando el factor de edad del ejercicio."""

    return preu_base * (1 + edat / 100)

In [37]:
def processar_clients_seguro(clients, nivells_seguro):
    """Añade categoría, precios y probabilidad de fraude a copias de los clientes."""

    resultat = []

    for client in clients:
        dades = client.copy()
        nivell = obtenir_nivell_seguro(client['codi'], nivells_seguro)
        dades['categoria'] = nivell['categoria']
        dades['preu_base'] = nivell['preu_base']
        dades['probabilitat_frau'] = nivell['fraude']
        dades['preu_final'] = calcular_preu_final(dades['preu_base'], dades['edat'])
        resultat.append(dades)

    return resultat

In [36]:
def ordenar_clients_seguro(clients):
    """Ordena por fraude descendente y, en los empates, por precio descendente."""

    return sorted(clients, key=lambda client: (client['probabilitat_frau'], client['preu_final']), reverse=True)

In [26]:
def mostrar_clients_seguro(clients):
    """Muestra las pólizas con precios a dos decimales y fraude en porcentaje."""
    for client in clients:
        print(f"{client['nom']} | Edad: {client['edat']} | Categoría: {client['categoria']} | Precio base: {client['preu_base']:.2f} € | Precio final: {client['preu_final']:.2f} € | Probabilidad de fraude: {client['probabilitat_frau']:.0%}")

In [31]:
def analitzar_clients_seguro(clients):
    """Presenta los máximos de fraude y precio y la prioridad de revisión."""

    major_frau = max(clients, key=lambda client: client['probabilitat_frau'])
    major_preu = max(clients, key=lambda client: client['preu_final'])
    prioritari = ordenar_clients_seguro(clients)[0]

    print('\nANÁLISIS')
    print(f"Mayor probabilidad de fraude: {major_frau['nom']} ({major_frau['probabilitat_frau']:.0%}).")
    print(f"Precio final más alto: {major_preu['nom']} ({major_preu['preu_final']:.2f} €).")
    print(f"Prioridad de revisión manual: {prioritari['nom']}, según la probabilidad de fraude asignada por el sistema; en caso de empate, se prioriza el precio final más alto.")

In [41]:
def main_seguro(clients, nivells_seguro):
    """Coordina el análisis de seguros y devuelve un diccionario por posición."""

    processats = processar_clients_seguro(clients, nivells_seguro)
    ordenats = ordenar_clients_seguro(processats)

    print('CLIENTES ORDENADOS POR FRAUDE Y PRECIO DESCENDENTES')
    mostrar_clients_seguro(ordenats)

    print("""

    **********

    """)

    analitzar_clients_seguro(processats)

    resultat = {}
    for posicio, client in enumerate(ordenats, start=1):
        resultat[posicio] = client

    return resultat

In [42]:
resultat_seguro = main_seguro(clients_seguro, nivells_seguro)

CLIENTES ORDENADOS POR FRAUDE Y PRECIO DESCENDENTES
Isabel | Edad: 55 | Categoría: Risc crític | Precio base: 500.00 € | Precio final: 775.00 € | Probabilidad de fraude: 30%
Carlos | Edad: 30 | Categoría: Risc alt | Precio base: 350.00 € | Precio final: 455.00 € | Probabilidad de fraude: 15%
Jorge | Edad: 40 | Categoría: Risc mitjà | Precio base: 200.00 € | Precio final: 280.00 € | Probabilidad de fraude: 5%
Ana | Edad: 42 | Categoría: Baix risc | Precio base: 120.00 € | Precio final: 170.40 € | Probabilidad de fraude: 1%
Marta | Edad: 28 | Categoría: Baix risc | Precio base: 120.00 € | Precio final: 153.60 € | Probabilidad de fraude: 1%

    
    **********
    
    

ANÁLISIS
Mayor probabilidad de fraude: Isabel (30%).
Precio final más alto: Isabel (775.00 €).
Prioridad de revisión manual: Isabel, según la probabilidad de fraude asignada por el sistema; en caso de empate, se prioriza el precio final más alto.


**Resultado:** Isabel presenta la mayor probabilidad asignada (30 %) y el mayor precio (775,00 €), por lo que es la primera para revisión según la regla elegida. Ana y Marta empatan al 1 %; Ana va antes por su precio de 170,40 €, frente a 153,60 €.

## Nivell 3

### Exercici 1: Generador de contrasenyes segures

#### Algoritmo
1. `obtenir_grups` reúne los tipos seleccionados.
2. `validar_configuracio` exige longitud entera positiva, al menos un grupo y espacio para todos ellos.
3. `seleccionar_caracters` escoge un carácter por grupo y completa con caracteres del conjunto permitido.
4. `barrejar_caracters` mezcla la lista para que los tipos no ocupen posiciones fijas y la une como texto.
5. `crear_contrasenya` coordina y devuelve la cadena con los parámetros obligatorios `majuscules=True`, `minuscules=True`, `numeros=True`, `signes=False`.

**Alcance:** NumPy cumple el objetivo académico de practicar aleatoriedad, pero su generador no es criptográficamente seguro. La longitud solicitada y la presencia de tipos no garantizan por sí solas seguridad real.

Las pruebas muestran resultados de validación e imprimen las cadenas generadas.

In [58]:
def obtenir_grups(majuscules, minuscules, numeros, signes):
    """Reúne los conjuntos de caracteres habilitados por los parámetros booleanos."""

    grups = []
    if majuscules:
        grups.append(string.ascii_uppercase)
    if minuscules:
        grups.append(string.ascii_lowercase)
    if numeros:
        grups.append(string.digits)
    if signes:
        grups.append(string.punctuation)

    return grups

In [59]:
def validar_configuracio(longitud, grups):
    """Valida longitud y grupos; devuelve True o lanza ValueError."""

    if type(longitud) is not int:
        raise ValueError('La longitud debe ser un número entero.')
    if longitud <= 0:
        raise ValueError('La longitud debe ser mayor que cero.')
    if not grups:
        raise ValueError('Selecciona al menos un tipo de carácter.')
    if longitud < len(grups):
        raise ValueError('La longitud debe permitir un carácter de cada tipo seleccionado.')

    return True

In [65]:
def seleccionar_caracters(longitud, grups, rng):
    """Garantiza un carácter por grupo y completa la longitud aleatoriamente."""

    caracters = []

    for grup in grups:
        caracters.append(rng.choice(list(grup)))
    disponibles = ''.join(grups)
    while len(caracters) < longitud:
        caracters.append(rng.choice(list(disponibles)))

    return caracters

In [60]:
def barrejar_caracters(caracters, rng):
    """Mezcla los caracteres recibidos y devuelve la cadena resultante."""

    rng.shuffle(caracters)

    return ''.join(caracters)

In [62]:
def crear_contrasenya(longitud, majuscules=True, minuscules=True, numeros=True, signes=False):
    """Coordina la generación académica con NumPy y devuelve una cadena."""

    grups = obtenir_grups(majuscules, minuscules, numeros, signes)
    validar_configuracio(longitud, grups)
    rng = np.random.default_rng()
    caracters = seleccionar_caracters(longitud, grups, rng)

    return barrejar_caracters(caracters, rng)

In [57]:
# Genera una contraseña de 12 caracteres usando los parámetros por defecto:
# mayúsculas, minúsculas y números, sin signos.
print("Opciones por defecto: mayúsculas, minúsculas y números.")
print(crear_contrasenya(12))

# Activa los cuatro tipos. Debe aparecer al menos un carácter de cada tipo.
print("\nLos cuatro tipos de caracteres:")
print(crear_contrasenya(
    12,
    majuscules=True,
    minuscules=True,
    numeros=True,
    signes=True
))

# Desactiva las letras y los signos para generar ocho dígitos.
print("\nSolo números:")
print(crear_contrasenya(
    8,
    majuscules=False,
    minuscules=False,
    numeros=True,
    signes=False
))

print("\nConfiguraciones inválidas:")

# Cada tupla contiene la longitud y un diccionario de opciones.
# Un diccionario vacío mantiene los parámetros por defecto.
casos_invalidos = [
    (2, {}),    # Longitud insuficiente para los tres tipos por defecto.
    (0, {}),    # La longitud debe ser mayor que cero.
    (3.5, {}),  # La longitud debe ser un número entero.
    (12, {
        "majuscules": False,
        "minuscules": False,
        "numeros": False,
        "signes": False
    })         # Debe seleccionarse al menos un tipo de carácter.
]

# Ejecuta cada caso inválido por separado.
for longitud, opciones in casos_invalidos:
    try:
        # **opciones pasa las claves del diccionario como argumentos:
        # por ejemplo, {"signes": True} equivale a signes=True.
        resultado = crear_contrasenya(longitud, **opciones)

        # Solo se muestra si el generador acepta una configuración inválida.
        print(f"Configuración aceptada inesperadamente: {resultado}")

    except ValueError as error:
        # Muestra el error previsto y permite continuar con el siguiente caso.
        print(error)

Opciones por defecto: mayúsculas, minúsculas y números.
6cv3Y0ZBOvkn

Los cuatro tipos de caracteres:
QKV4i4'!z4J<

Solo números:
17746328

Configuraciones inválidas:
La longitud debe permitir un carácter de cada tipo seleccionado.
La longitud debe ser mayor que cero.
La longitud debe ser un número entero.
Selecciona al menos un tipo de carácter.


**Resultado:** se generan y muestran tres contraseñas: una de 12 caracteres con las opciones por defecto, otra de 12 con los cuatro tipos de caracteres y una de 8 formada solo por números.

También se ejecutan cuatro configuraciones inválidas para observar los mensajes de error: longitud insuficiente, cero, longitud decimal y ningún tipo de carácter seleccionado.

### Exercici 2: Processament automàtic de dades esportives

#### Algoritmo
1. `llegir_partits` lee el TXT en UTF-8, ignora líneas vacías, separa campos por tabulaciones y convierte goles a enteros.
2. `crear_estadistiques` inicializa goles a favor y en contra, victorias, empates, derrotas y puntos.
3. `actualitzar_equip` acumula los goles y asigna 3 puntos por victoria, 1 por empate y 0 por derrota; devuelve las estadísticas actualizadas.
4. `processar_partits` aplica la actualización a local y visitante, invirtiendo sus goles, para todos los partidos.
5. `ordenar_classificacio` ordena por puntos descendentes y, solo para presentar empates, por nombre alfabético.
6. `mostrar_estadistiques` presenta la tabla y `mostrar_resum_esportiu` muestra los máximos de goles y la clasificación.
7. `main_esports` coordina y devuelve el diccionario final.

**Decisiones:** el separador es `\t`, porque los nombres de equipos contienen espacios. Se procesan todas las filas del histórico sin deduplicar enfrentamientos; un mismo par de equipos puede jugar varias veces. La ordenación alfabética de los empates no pretende ser un reglamento deportivo. Se muestran todos los equipos que compartan un máximo. El archivo debe conservarse junto a la entrega.

In [69]:
def llegir_partits(ruta):
    """Lee los partidos separados por tabulaciones."""

    partits = []

    with open(ruta, 'r', encoding='utf-8-sig') as fitxer:
        for linia in fitxer:
            if linia.strip():
                local, marcador, visitant = linia.strip().split('\t')
                gols_local, gols_visitant = marcador.split('-')
                partits.append({'local': local.strip(), 'visitant': visitant.strip(), 'gols_local': int(gols_local), 'gols_visitant': int(gols_visitant)})

    return partits

In [71]:
def crear_estadistiques():
    """Crea las estadísticas iniciales de un equipo."""

    return {'gols_favor': 0, 'gols_contra': 0, 'guanyats': 0, 'empatats': 0, 'perduts': 0, 'punts': 0}

In [72]:
def actualitzar_equip(dades, gols_favor, gols_contra):
    """Actualiza las estadísticas de un equipo con un partido."""

    dades['gols_favor'] += gols_favor
    dades['gols_contra'] += gols_contra

    if gols_favor > gols_contra:
        dades['guanyats'] += 1
        dades['punts'] += 3
    elif gols_favor == gols_contra:
        dades['empatats'] += 1
        dades['punts'] += 1
    else:
        dades['perduts'] += 1

    return dades

In [74]:
def processar_partits(partits):
    """Acumula las estadísticas de todos los equipos."""

    equips = {}

    for partit in partits:
        local = partit['local']
        visitant = partit['visitant']
        if local not in equips:
            equips[local] = crear_estadistiques()
        if visitant not in equips:
            equips[visitant] = crear_estadistiques()
        equips[local] = actualitzar_equip(equips[local], partit['gols_local'], partit['gols_visitant'])
        equips[visitant] = actualitzar_equip(equips[visitant], partit['gols_visitant'], partit['gols_local'])

    return equips

In [75]:
def ordenar_classificacio(equips):
    """Ordena por puntos descendentes y nombre en caso de empate."""

    ordenats = sorted(equips.items(), key=lambda element: (-element[1]['punts'], element[0]))

    return dict(ordenats)

In [76]:
def mostrar_estadistiques(equips):
    """Muestra todas las estadísticas acumuladas."""

    print('\nESTADÍSTICAS POR EQUIPO')
    print(f"{'Equipo':<20} {'GF':>5} {'GC':>5} {'V':>4} {'E':>4} {'D':>4} {'Pts':>5}")

    for nom, dades in equips.items():
        print(f"{nom:<20} {dades['gols_favor']:>5} {dades['gols_contra']:>5} {dades['guanyats']:>4} {dades['empatats']:>4} {dades['perduts']:>4} {dades['punts']:>5}")

In [78]:
def mostrar_resum_esportiu(equips):
    """Muestra los máximos de goles y la clasificación."""

    maxim_favor = max((d['gols_favor'] for d in equips.values()))
    maxim_contra = max((d['gols_contra'] for d in equips.values()))

    for nom, dades in equips.items():
        if dades['gols_favor'] == maxim_favor:
            print(f'\nEquipo más goleador: {nom}, con {maxim_favor} goles.')
        if dades['gols_contra'] == maxim_contra:
            print(f'\nEquipo más goleado: {nom}, con {maxim_contra} goles.')

    print('\nCLASIFICACIÓN FINAL')

    for posicio, (nom, dades) in enumerate(equips.items(), start=1):
        print(f"{posicio}. {nom}: {dades['punts']} puntos")

In [66]:
def main_esports(ruta):
    """Coordina la lectura, el cálculo y la presentación."""

    partits = llegir_partits(ruta)
    equips = processar_partits(partits)
    classificacio = ordenar_classificacio(equips)

    print(f'Partidos procesados: {len(partits)}')
    print(f'Equipos: {len(equips)}')

    mostrar_estadistiques(classificacio)
    mostrar_resum_esportiu(classificacio)

    return classificacio

In [79]:
ruta_partidos = Path("historic partits.txt")

if not ruta_partidos.is_file():
    ruta_partidos = Path("/content/historic partits.txt")
if not ruta_partidos.is_file():
    raise FileNotFoundError("Coloca historic partits.txt junto al notebook o súbelo a /content en Colab.")

classificacio = main_esports(ruta_partidos)

Partidos procesados: 500
Equipos: 20

ESTADÍSTICAS POR EQUIPO
Equipo                  GF    GC    V    E    D   Pts
Girona FC              139    94   31    3   13    96
Llagostera             159   142   29    7   20    94
Sabadell               141   121   26    7   15    85
Cornellà               147   146   25    7   22    82
RCD Espanyol           131   144   23   11   21    80
Figueres               161   154   23   10   23    79
Lleida Esportiu        129   133   23    6   23    75
Terrassa               147   164   20   14   23    74
FC Barcelona           125   115   22    7   15    73
Vilafranca             157   172   20   12   25    72
Badalona               134   124   18   14   16    68
Nàstic de Tarragona    130   148   20    8   27    68
Reus Deportiu          119   111   20    8   16    68
Granollers             122   117   21    4   24    67
Olot                   132   144   18   10   26    64
Sant Andreu            123   134   17   11   25    62
Manlleu             

**Resultado:** se procesan 500 partidos de 20 equipos. Figueres es el más goleador (161) y Vilafranca el más goleado (172). Girona FC lidera con 96 puntos, seguido de Llagostera con 94 y Sabadell con 85. Goles a favor y en contra suman ambos 2.618; las participaciones suman 1.000, dos por partido.